In [1]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import time

from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV
from sklearn.discriminant_analysis import (LinearDiscriminantAnalysis,
                                           QuadraticDiscriminantAnalysis)
from sklearn.ensemble import (AdaBoostClassifier, BaggingClassifier,
                               GradientBoostingClassifier,
                               RandomForestClassifier, StackingClassifier)
from sklearn.linear_model import LogisticRegression, Perceptron
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score
from sklearn.model_selection import StratifiedKFold, learning_curve
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except Exception:
    XGBOOST_AVAILABLE = False

sns.set_theme(style="whitegrid")

PROJECT_ROOT  = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

# Output folder
LC_DIR    = PROJECT_ROOT / "outputs" / "learning_curves"
TABLE_DIR = PROJECT_ROOT / "outputs" / "comparison_tables"
for p in [LC_DIR, TABLE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Learning curve outputs →", LC_DIR)

Learning curve outputs → c:\Users\ssath\OneDrive\Documents\PCOS detection using ML\pcos-prediction-ml-clean\outputs\learning_curves


In [12]:
# ── with FE (11 features) ──────────────────────────────────────────────────
X_train_fe = pd.read_csv(PROCESSED_DIR / "X_train_selected_imp.csv")
y_train_fe = pd.read_csv(PROCESSED_DIR / "y_train.csv")["PCOS"]

# ── without FE (41 features) ──────────────────────────────────────────────
X_train_no = pd.read_csv(PROCESSED_DIR / "X_train_processed.csv")
y_train_no = pd.read_csv(PROCESSED_DIR / "y_train.csv")["PCOS"]

# ── Test set (30% — only used for final dot on curve) ─────────────────────
X_test     = pd.read_csv(PROCESSED_DIR / "X_test_processed.csv")
X_test_fe  = pd.read_csv(PROCESSED_DIR / "X_test_selected_imp.csv")
y_test     = pd.read_csv(PROCESSED_DIR / "y_test.csv")["PCOS"]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Training sizes to try: 10% to 100% of X_train in steps
# Replace 1.00 with 0.99 to avoid exact n_samples = total error
TRAIN_SIZES = np.array([0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90, 0.99])

print(f"with FE    train shape : {X_train_fe.shape}")
print(f"without FE train shape : {X_train_no.shape}")
print(f"Test shape             : {X_test.shape}")
print(f"Train sizes to try     : {TRAIN_SIZES}")

with FE    train shape : (378, 11)
without FE train shape : (378, 41)
Test shape             : (163, 41)
Train sizes to try     : [0.1  0.2  0.3  0.4  0.5  0.6  0.7  0.8  0.9  0.99]


In [16]:
def get_models():
    models = {
        "Logistic Regression": LogisticRegression(
            max_iter=3000, class_weight="balanced", random_state=42),

        "Decision Tree": DecisionTreeClassifier(
            class_weight="balanced", random_state=42),

        "Random Forest": RandomForestClassifier(
            n_estimators=300, class_weight="balanced", random_state=42),

        "SVM": SVC(
            kernel="rbf", probability=True,
            class_weight="balanced", random_state=42),

        "Gaussian NB": GaussianNB(),

        "Bagging": BaggingClassifier(
            estimator=DecisionTreeClassifier(
                class_weight="balanced", random_state=42),
            n_estimators=150, random_state=42),

        "AdaBoost": AdaBoostClassifier(
            n_estimators=150, learning_rate=0.5, random_state=42),

        "Gradient Boosting": GradientBoostingClassifier(random_state=42),

        "KNN": KNeighborsClassifier(n_neighbors=21),

        "LDA": LinearDiscriminantAnalysis(),

        "QDA": QuadraticDiscriminantAnalysis(reg_param=0.1),

        "Perceptron": CalibratedClassifierCV(
            Perceptron(class_weight="balanced", random_state=42), cv=5),
    }

    if XGBOOST_AVAILABLE:
        models["XGBoost"] = XGBClassifier(
            n_estimators=250, max_depth=3, learning_rate=0.05,
            subsample=0.9, colsample_bytree=0.9,
            eval_metric="logloss", random_state=42)
    else:
        models["XGBoost"] = GradientBoostingClassifier(random_state=43)

    return models

# We call get_models() fresh each time so FE and no-FE
# get completely independent model objects
print("Model factory defined.")
print(f"Total models: {len(get_models())}")

Model factory defined.
Total models: 13


In [19]:
def compute_learning_curve(model, X_train, y_train,
                            X_test, y_test, train_sizes):
    """
    For each training size:
      1. Take a stratified subset of X_train
      2. Fit the model on that subset
      3. Record:
           - train F1, train AUC  (how well it fits its own data)
           - val F1,   val AUC    (5-fold CV on the subset)
           - test F1,  test AUC   (on held-out 30% test set)

    This gives THREE curves per metric:
      Train  → always high (memorisation)
      Val    → rises then plateaus (generalisation on CV)
      Test   → rises then plateaus (true generalisation)

    The GAP between Train and Val/Test = overfitting
    When Val and Test curves are CLOSE = model is stable
    """
    n_total = len(X_train)

    records = []
    for size in train_sizes:
        n_samples = max(int(n_total * size), 10)

        # ── Stratified subset ─────────────────────────────────────────
        from sklearn.model_selection import StratifiedShuffleSplit
        sss = StratifiedShuffleSplit(
            n_splits=1, train_size=n_samples, random_state=42)
        idx, _ = next(sss.split(X_train, y_train))

        X_sub = X_train.iloc[idx]
        y_sub = y_train.iloc[idx]

        # ── Fit ───────────────────────────────────────────────────────
        m = clone(model)
        t0 = time.time()
        m.fit(X_sub, y_sub)
        fit_time = round(time.time() - t0, 4)

        # ── Train scores (on the subset itself) ───────────────────────
        if hasattr(m, "predict_proba"):
            train_proba = m.predict_proba(X_sub)[:, 1]
            test_proba  = m.predict_proba(X_test)[:, 1]
        elif hasattr(m, "decision_function"):
            s = m.decision_function(X_sub)
            train_proba = (s - s.min()) / (s.max() - s.min() + 1e-12)
            s2 = m.decision_function(X_test)
            test_proba  = (s2 - s2.min()) / (s2.max() - s2.min() + 1e-12)
        else:
            train_proba = m.predict(X_sub).astype(float)
            test_proba  = m.predict(X_test).astype(float)

        train_preds = (train_proba >= 0.5).astype(int)
        test_preds  = (test_proba  >= 0.5).astype(int)

        # ── 5-fold CV F1 on the subset ────────────────────────────────
        cv_inner = StratifiedKFold(n_splits=min(5, y_sub.value_counts().min()),
                                   shuffle=True, random_state=42)
        cv_f1_scores, cv_auc_scores = [], []
        for tr, va in cv_inner.split(X_sub, y_sub):
            mc = clone(model)
            mc.fit(X_sub.iloc[tr], y_sub.iloc[tr])
            if hasattr(mc, "predict_proba"):
                sc = mc.predict_proba(X_sub.iloc[va])[:, 1]
            else:
                sc = mc.predict(X_sub.iloc[va]).astype(float)
            preds_va = (sc >= 0.5).astype(int)
            cv_f1_scores.append(
                f1_score(y_sub.iloc[va], preds_va, zero_division=0))
            try:
                cv_auc_scores.append(roc_auc_score(y_sub.iloc[va], sc))
            except Exception:
                cv_auc_scores.append(np.nan)

        records.append({
            "train_size_pct" : round(size * 100, 0),
            "train_size_n"   : n_samples,
            "fit_time_sec"   : fit_time,
            # Train (memorisation)
            "train_f1"       : f1_score(y_sub, train_preds, zero_division=0),
            "train_auc"      : roc_auc_score(y_sub, train_proba),
            "train_acc"      : accuracy_score(y_sub, train_preds),
            # Validation CV (generalisation estimate)
            "val_f1_mean"    : np.nanmean(cv_f1_scores),
            "val_f1_std"     : np.nanstd(cv_f1_scores),
            "val_auc_mean"   : np.nanmean(cv_auc_scores),
            "val_auc_std"    : np.nanstd(cv_auc_scores),
            # Test (true generalisation — 30% holdout)
            "test_f1"        : f1_score(y_test, test_preds, zero_division=0),
            "test_auc"       : roc_auc_score(y_test, test_proba),
            "test_acc"       : accuracy_score(y_test, test_preds),
        })

        print(f"    size={int(size*100):3d}% (n={n_samples:4d}) | "
              f"TrainF1={records[-1]['train_f1']:.4f} | "
              f"ValF1={records[-1]['val_f1_mean']:.4f} | "
              f"TestF1={records[-1]['test_f1']:.4f} | "
              f"Time={fit_time:.3f}s")

    return pd.DataFrame(records)

print("Learning curve function defined.")

Learning curve function defined.


In [20]:
def plot_learning_curve(lc_df, model_name, experiment_label, out_dir):
    """
    Produces 2 side-by-side plots:
      Left  → F1 Score learning curve
      Right → ROC-AUC learning curve

    Three lines each:
      Red    = Train  (memorisation — always high)
      Orange = Val CV (generalisation estimate)
      Green  = Test   (true generalisation on 30% holdout)

    Shaded band = ± std of CV folds

    What to look for:
      - Gap between red and green/orange = overfitting
      - Green and orange converging = stable model
      - All lines still rising at 100% = more data would help
      - All lines flat at 100% = model has saturated
    """
    sizes = lc_df["train_size_pct"].values

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for ax, metric, title in [
        (axes[0], "f1",  "F1 Score"),
        (axes[1], "auc", "ROC-AUC"),
    ]:
        train_vals = lc_df[f"train_{metric}"].values
        val_mean   = lc_df[f"val_{metric}_mean"].values
        val_std    = lc_df[f"val_{metric}_std"].values
        test_vals  = lc_df[f"test_{metric}"].values

        # Train line
        ax.plot(sizes, train_vals, "o-", color="#d62728",
                linewidth=2, markersize=6,
                label="Train (memorisation)")

        # Val CV line with shaded std band
        ax.plot(sizes, val_mean, "s-", color="#ff7f0e",
                linewidth=2, markersize=6,
                label="Val CV (generalisation estimate)")
        ax.fill_between(sizes,
                        val_mean - val_std,
                        val_mean + val_std,
                        alpha=0.15, color="#ff7f0e")

        # Test line
        ax.plot(sizes, test_vals, "^-", color="#2ca02c",
                linewidth=2, markersize=6,
                label="Test 30% (true generalisation)")

        # Annotate test line values
        for x, y in zip(sizes, test_vals):
            ax.annotate(f"{y:.4f}", (x, y),
                        textcoords="offset points",
                        xytext=(0, 8), ha="center",
                        fontsize=7.5, color="#2ca02c")

        ax.set_xlabel("Training Data Size (%)", fontsize=10)
        ax.set_ylabel(title, fontsize=10)
        ax.set_xticks(sizes)
        ax.set_xticklabels([f"{int(s)}%" for s in sizes], fontsize=9)
        ax.set_ylim(
            max(0, min(np.nanmin(val_mean), np.nanmin(test_vals)) - 0.10),
            1.08
        )
        ax.set_title(f"{title} Learning Curve", fontsize=11, fontweight="bold")
        ax.legend(fontsize=9, loc="lower right")
        ax.grid(True, linestyle="--", alpha=0.4)

    fig.suptitle(
        f"{experiment_label} — {model_name}\n"
        f"Learning Curve: How performance improves with more training data",
        fontsize=12, fontweight="bold"
    )
    plt.tight_layout()

    safe = (model_name.lower()
            .replace(" ", "_").replace("/", "_")
            .replace("(", "").replace(")", ""))
    path = out_dir / f"learning_curve_{safe}.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved → {path}")
    return path


def plot_fe_vs_nofe_comparison(lc_fe, lc_no, model_name, out_dir):
    """
    Side by side:
      Left  → F1:     with FE vs without FE (test line only)
      Right → ROC-AUC: with FE vs without FE (test line only)

    This is your KEY graph proving FE is better:
      - with FE reaches high F1 FASTER (steeper curve)
      - with FE final score >= without FE
      - with FE trains FASTER (shown in title)
    """
    sizes = lc_fe["train_size_pct"].values

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for ax, metric, title in [
        (axes[0], "f1",  "F1 Score"),
        (axes[1], "auc", "ROC-AUC"),
    ]:
        fe_test  = lc_fe[f"test_{metric}"].values
        no_test  = lc_no[f"test_{metric}"].values
        fe_val   = lc_fe[f"val_{metric}_mean"].values
        no_val   = lc_no[f"val_{metric}_mean"].values

        # with FE lines
        ax.plot(sizes, fe_test, "o-", color="#1f77b4",
                linewidth=2.2, markersize=7,
                label="with FE — Test (11 features)")
        ax.plot(sizes, fe_val,  "o--", color="#1f77b4",
                linewidth=1.4, markersize=5, alpha=0.6,
                label="with FE — Val CV")

        # without FE lines
        ax.plot(sizes, no_test, "s-", color="#d62728",
                linewidth=2.2, markersize=7,
                label="without FE — Test (41 features)")
        ax.plot(sizes, no_val,  "s--", color="#d62728",
                linewidth=1.4, markersize=5, alpha=0.6,
                label="without FE — Val CV")

        # Annotate final point
        ax.annotate(f"FE: {fe_test[-1]:.4f}",
                    (sizes[-1], fe_test[-1]),
                    textcoords="offset points", xytext=(5, 5),
                    fontsize=8, color="#1f77b4", fontweight="bold")
        ax.annotate(f"No FE: {no_test[-1]:.4f}",
                    (sizes[-1], no_test[-1]),
                    textcoords="offset points", xytext=(5, -12),
                    fontsize=8, color="#d62728", fontweight="bold")

        ax.set_xlabel("Training Data Size (%)", fontsize=10)
        ax.set_ylabel(title, fontsize=10)
        ax.set_xticks(sizes)
        ax.set_xticklabels([f"{int(s)}%" for s in sizes], fontsize=9)
        ax.set_ylim(
            max(0, min(np.nanmin(fe_val), np.nanmin(no_val)) - 0.10),
            1.10
        )
        ax.set_title(f"{title}: with FE vs without FE", fontsize=11,
                     fontweight="bold")
        ax.legend(fontsize=8, loc="lower right")
        ax.grid(True, linestyle="--", alpha=0.4)

    fig.suptitle(
        f"with FE vs without FE — {model_name}\n"
        f"Learning Curve Comparison (11 features vs 41 features)",
        fontsize=12, fontweight="bold"
    )
    plt.tight_layout()

    safe = (model_name.lower()
            .replace(" ", "_").replace("/", "_")
            .replace("(", "").replace(")", ""))
    path = out_dir / f"fe_vs_nofe_{safe}.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved → {path}")

print("Plot functions defined.")

Plot functions defined.


In [21]:
# Storage for summary
all_results_fe = {}
all_results_no = {}

models_fe = get_models()
models_no = get_models()

for name in models_fe.keys():
    print(f"\n{'='*60}")
    print(f"Model: {name}")
    print(f"{'='*60}")

    # ── Per-model output folder ────────────────────────────────────
    m_dir = LC_DIR / (name.lower()
                      .replace(" ", "_")
                      .replace("(", "").replace(")", ""))
    m_dir.mkdir(parents=True, exist_ok=True)

    # ── with FE ───────────────────────────────────────────────────
    print(f"  → with FE (11 features):")
    try:
        lc_fe = compute_learning_curve(
            model      = models_fe[name],
            X_train    = X_train_fe,
            y_train    = y_train_fe,
            X_test     = X_test_fe,
            y_test     = y_test,
            train_sizes= TRAIN_SIZES
        )
        lc_fe["model"]      = name
        lc_fe["experiment"] = "with_FE"
        all_results_fe[name] = lc_fe

        # Individual model plot — with FE
        plot_learning_curve(lc_fe, name, "with FE", m_dir)

    except Exception as e:
        print(f"  ✗ with FE failed: {e}")
        lc_fe = None

    # ── without FE ────────────────────────────────────────────────
    print(f"  → without FE (41 features):")
    try:
        lc_no = compute_learning_curve(
            model      = models_no[name],
            X_train    = X_train_no,
            y_train    = y_train_no,
            X_test     = X_test,
            y_test     = y_test,
            train_sizes= TRAIN_SIZES
        )
        lc_no["model"]      = name
        lc_no["experiment"] = "without_FE"
        all_results_no[name] = lc_no

        # Individual model plot — without FE
        plot_learning_curve(lc_no, name, "without FE", m_dir)

    except Exception as e:
        print(f"  ✗ without FE failed: {e}")
        lc_no = None

    # ── Comparison plot — with FE vs without FE ───────────────────
    if lc_fe is not None and lc_no is not None:
        plot_fe_vs_nofe_comparison(lc_fe, lc_no, name, m_dir)

    print(f"  ✓ {name} complete")

print("\n✓ All individual model learning curves complete.")


Model: Logistic Regression
  → with FE (11 features):
    size= 10% (n=  37) | TrainF1=0.9600 | ValF1=0.8200 | TestF1=0.7551 | Time=0.008s
    size= 20% (n=  75) | TrainF1=0.8679 | ValF1=0.6889 | TestF1=0.8214 | Time=0.007s
    size= 30% (n= 113) | TrainF1=0.8354 | ValF1=0.8006 | TestF1=0.8000 | Time=0.005s
    size= 40% (n= 151) | TrainF1=0.8411 | ValF1=0.7770 | TestF1=0.8673 | Time=0.006s
    size= 50% (n= 189) | TrainF1=0.8682 | ValF1=0.8114 | TestF1=0.8545 | Time=0.005s
    size= 60% (n= 226) | TrainF1=0.8387 | ValF1=0.8067 | TestF1=0.8571 | Time=0.005s
    size= 70% (n= 264) | TrainF1=0.8634 | ValF1=0.8315 | TestF1=0.8649 | Time=0.005s
    size= 80% (n= 302) | TrainF1=0.8599 | ValF1=0.8208 | TestF1=0.8545 | Time=0.006s
    size= 90% (n= 340) | TrainF1=0.8718 | ValF1=0.8567 | TestF1=0.8545 | Time=0.006s
    size= 99% (n= 374) | TrainF1=0.8775 | ValF1=0.8641 | TestF1=0.8545 | Time=0.006s
  Saved → c:\Users\ssath\OneDrive\Documents\PCOS detection using ML\pcos-prediction-ml-clean\ou

In [22]:
def plot_all_models_summary(all_results, experiment_label, metric,
                             metric_label, out_dir):
    """
    One line per model showing test performance vs training size.
    Lets you compare ALL models at once.
    Shows which model improves most with more data.
    """
    fig, ax = plt.subplots(figsize=(14, 6))

    cmap   = plt.cm.tab20
    colors = [cmap(i / len(all_results)) for i in range(len(all_results))]

    for (name, lc_df), color in zip(all_results.items(), colors):
        sizes = lc_df["train_size_pct"].values
        vals  = lc_df[f"test_{metric}"].values
        ax.plot(sizes, vals, "o-", color=color,
                linewidth=1.8, markersize=5, label=name)
        # Annotate final value
        ax.annotate(f"{vals[-1]:.3f}",
                    (sizes[-1], vals[-1]),
                    textcoords="offset points",
                    xytext=(5, 0), fontsize=7,
                    color=color)

    ax.set_xlabel("Training Data Size (%)", fontsize=11)
    ax.set_ylabel(metric_label, fontsize=11)
    ax.set_xticks(TRAIN_SIZES * 100)
    ax.set_xticklabels([f"{int(s*100)}%" for s in TRAIN_SIZES], fontsize=9)
    ax.set_title(
        f"{experiment_label} — All Models\n"
        f"{metric_label} on Test Set vs Training Data Size",
        fontsize=12, fontweight="bold"
    )
    ax.legend(fontsize=8, loc="lower right",
              ncol=2, bbox_to_anchor=(1.0, 0.0))
    ax.grid(True, linestyle="--", alpha=0.4)
    plt.tight_layout()

    path = out_dir / f"all_models_{experiment_label}_{metric}.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved → {path}")


# with FE — all models
plot_all_models_summary(all_results_fe, "with_FE",    "f1",  "F1 Score", LC_DIR)
plot_all_models_summary(all_results_fe, "with_FE",    "auc", "ROC-AUC",  LC_DIR)

# without FE — all models
plot_all_models_summary(all_results_no, "without_FE", "f1",  "F1 Score", LC_DIR)
plot_all_models_summary(all_results_no, "without_FE", "auc", "ROC-AUC",  LC_DIR)

print("Summary plots done.")

Saved → c:\Users\ssath\OneDrive\Documents\PCOS detection using ML\pcos-prediction-ml-clean\outputs\learning_curves\all_models_with_FE_f1.png
Saved → c:\Users\ssath\OneDrive\Documents\PCOS detection using ML\pcos-prediction-ml-clean\outputs\learning_curves\all_models_with_FE_auc.png
Saved → c:\Users\ssath\OneDrive\Documents\PCOS detection using ML\pcos-prediction-ml-clean\outputs\learning_curves\all_models_without_FE_f1.png
Saved → c:\Users\ssath\OneDrive\Documents\PCOS detection using ML\pcos-prediction-ml-clean\outputs\learning_curves\all_models_without_FE_auc.png
Summary plots done.


In [23]:
# Combine all into one big CSV
all_rows = []
for name, lc_df in all_results_fe.items():
    all_rows.append(lc_df)
for name, lc_df in all_results_no.items():
    all_rows.append(lc_df)

df_all = pd.concat(all_rows, ignore_index=True)
df_all.to_csv(LC_DIR   / "learning_curve_all_results.csv", index=False)
df_all.to_csv(TABLE_DIR/ "learning_curve_all_results.csv", index=False)

display(df_all.head(20))
print(f"\nTotal rows saved: {len(df_all)}")

,train_size_pct,train_size_n,fit_time_sec,train_f1,train_auc,train_acc,val_f1_mean,val_f1_std,val_auc_mean,val_auc_std,test_f1,test_auc,test_acc,model,experiment
0,10.0,37,0.0083,0.960000,0.996667,0.972973,0.820000,0.183303,0.953333,0.058119,0.755102,0.907890,0.852761,Logistic Regression,with_FE
1,20.0,75,0.0072,0.867925,0.969600,0.906667,0.688889,0.143157,0.856000,0.109836,0.821429,0.939794,0.877301,Logistic Regression,with_FE
2,30.0,113,0.0052,0.835443,0.971550,0.884956,0.800556,0.041515,0.943690,0.021321,0.800000,0.932247,0.865031,Logistic Regression,with_FE
3,40.0,151,0.0064,0.841121,0.966931,0.887417,0.776970,0.055599,0.950333,0.022146,0.867257,0.946484,0.907975,Logistic Regression,with_FE
4,50.0,189,0.0047,0.868217,0.972187,0.910053,0.811442,0.039378,0.945538,0.026106,0.854545,0.950257,0.901840,Logistic Regression,with_FE
5,60.0,226,0.0053,0.838710,0.958037,0.889381,0.806668,0.100140,0.932033,0.037549,0.857143,0.949400,0.901840,Logistic Regression,with_FE
6,70.0,264,0.0047,0.863388,0.959478,0.905303,0.831478,0.035585,0.947470,0.015954,0.864865,0.951801,0.907975,Logistic Regression,with_FE
7,80.0,302,0.0058,0.859903,0.963378,0.903974,0.820762,0.033995,0.944197,0.032662,0.854545,0.950086,0.901840,Logistic Regression,with_FE
8,90.0,340,0.0056,0.871795,0.965891,0.911765,0.856673,0.064947,0.956456,0.034318,0.854545,0.950429,0.901840,Logistic Regression,with_FE
9,99.0,374,0.0056,0.877470,0.966378,0.917112,0.864118,0.064122,0.959127,0.022070,0.854545,0.949400,0.901840,Logistic Regression,with_FE



Total rows saved: 240


In [24]:
def plot_all_models_summary(all_results, experiment_label, metric,
                             metric_label, out_dir):
    """
    One line per model showing test performance vs training size.
    Lets you compare ALL models at once.
    Shows which model improves most with more data.
    """
    fig, ax = plt.subplots(figsize=(14, 6))

    cmap   = plt.cm.tab20
    colors = [cmap(i / len(all_results)) for i in range(len(all_results))]

    for (name, lc_df), color in zip(all_results.items(), colors):
        sizes = lc_df["train_size_pct"].values
        vals  = lc_df[f"test_{metric}"].values
        ax.plot(sizes, vals, "o-", color=color,
                linewidth=1.8, markersize=5, label=name)
        # Annotate final value
        ax.annotate(f"{vals[-1]:.3f}",
                    (sizes[-1], vals[-1]),
                    textcoords="offset points",
                    xytext=(5, 0), fontsize=7,
                    color=color)

    ax.set_xlabel("Training Data Size (%)", fontsize=11)
    ax.set_ylabel(metric_label, fontsize=11)
    ax.set_xticks(TRAIN_SIZES * 100)
    ax.set_xticklabels([f"{int(s*100)}%" for s in TRAIN_SIZES], fontsize=9)
    ax.set_title(
        f"{experiment_label} — All Models\n"
        f"{metric_label} on Test Set vs Training Data Size",
        fontsize=12, fontweight="bold"
    )
    ax.legend(fontsize=8, loc="lower right",
              ncol=2, bbox_to_anchor=(1.0, 0.0))
    ax.grid(True, linestyle="--", alpha=0.4)
    plt.tight_layout()

    path = out_dir / f"all_models_{experiment_label}_{metric}.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved → {path}")


# with FE — all models
plot_all_models_summary(all_results_fe, "with_FE",    "f1",  "F1 Score", LC_DIR)
plot_all_models_summary(all_results_fe, "with_FE",    "auc", "ROC-AUC",  LC_DIR)

# without FE — all models
plot_all_models_summary(all_results_no, "without_FE", "f1",  "F1 Score", LC_DIR)
plot_all_models_summary(all_results_no, "without_FE", "auc", "ROC-AUC",  LC_DIR)

print("Summary plots done.")

Saved → c:\Users\ssath\OneDrive\Documents\PCOS detection using ML\pcos-prediction-ml-clean\outputs\learning_curves\all_models_with_FE_f1.png
Saved → c:\Users\ssath\OneDrive\Documents\PCOS detection using ML\pcos-prediction-ml-clean\outputs\learning_curves\all_models_with_FE_auc.png
Saved → c:\Users\ssath\OneDrive\Documents\PCOS detection using ML\pcos-prediction-ml-clean\outputs\learning_curves\all_models_without_FE_f1.png
Saved → c:\Users\ssath\OneDrive\Documents\PCOS detection using ML\pcos-prediction-ml-clean\outputs\learning_curves\all_models_without_FE_auc.png
Summary plots done.


In [25]:
# ── Save all results CSV ───────────────────────────────────────────────────
all_rows = []
for name, lc_df in all_results_fe.items():
    all_rows.append(lc_df)
for name, lc_df in all_results_no.items():
    all_rows.append(lc_df)

if all_rows:
    df_all = pd.concat(all_rows, ignore_index=True)
    df_all.to_csv(LC_DIR    / "learning_curve_all_results.csv", index=False)
    df_all.to_csv(TABLE_DIR / "learning_curve_all_results.csv", index=False)
    print(f"All results saved: {len(df_all)} rows")
else:
    print("No results to save")

# ── Final summary table ────────────────────────────────────────────────────
summary_rows = []

all_model_names = set(list(all_results_fe.keys()) +
                       list(all_results_no.keys()))

for name in all_model_names:
    row = {"Model": name}

    if name in all_results_fe:
        lc_fe = all_results_fe[name]
        fe_last   = lc_fe.iloc[-1]
        row["with_FE_Test_F1"]    = round(fe_last["test_f1"],  4)
        row["with_FE_Test_AUC"]   = round(fe_last["test_auc"], 4)
        row["with_FE_Total_Time"] = round(lc_fe["fit_time_sec"].sum(), 4)
    else:
        row["with_FE_Test_F1"]    = np.nan
        row["with_FE_Test_AUC"]   = np.nan
        row["with_FE_Total_Time"] = np.nan

    if name in all_results_no:
        lc_no = all_results_no[name]
        no_last   = lc_no.iloc[-1]
        row["without_FE_Test_F1"]    = round(no_last["test_f1"],  4)
        row["without_FE_Test_AUC"]   = round(no_last["test_auc"], 4)
        row["without_FE_Total_Time"] = round(lc_no["fit_time_sec"].sum(), 4)
    else:
        row["without_FE_Test_F1"]    = np.nan
        row["without_FE_Test_AUC"]   = np.nan
        row["without_FE_Total_Time"] = np.nan

    row["F1_Gain_from_FE"]   = round(
        row["with_FE_Test_F1"] - row["without_FE_Test_F1"], 4)
    row["AUC_Gain_from_FE"]  = round(
        row["with_FE_Test_AUC"] - row["without_FE_Test_AUC"], 4)
    row["Time_Saved_by_FE"]  = round(
        row["without_FE_Total_Time"] - row["with_FE_Total_Time"], 4)

    summary_rows.append(row)

df_summary = (pd.DataFrame(summary_rows)
                .sort_values("F1_Gain_from_FE", ascending=False)
                .reset_index(drop=True))

df_summary.to_csv(LC_DIR    / "fe_vs_nofe_final_summary.csv", index=False)
df_summary.to_csv(TABLE_DIR / "fe_vs_nofe_final_summary.csv", index=False)

display(df_summary)
print("\n✓ All done. Everything saved under:", LC_DIR)

All results saved: 240 rows


,Model,with_FE_Test_F1,with_FE_Test_AUC,with_FE_Total_Time,without_FE_Test_F1,without_FE_Test_AUC,without_FE_Total_Time,F1_Gain_from_FE,AUC_Gain_from_FE,Time_Saved_by_FE
0,KNN,0.8085,0.9352,0.0129,0.7126,0.9352,0.0141,0.0959,0.0000,0.0012
1,Logistic Regression,0.8545,0.9494,0.0588,0.7679,0.9365,0.0856,0.0866,0.0129,0.0268
2,Gaussian NB,0.8333,0.9513,0.0148,0.7619,0.8824,0.0179,0.0714,0.0689,0.0031
3,SVM,0.8235,0.9431,0.0665,0.7921,0.9349,0.1108,0.0314,0.0082,0.0443
4,LDA,0.8200,0.9463,0.0168,0.7961,0.9484,0.0478,0.0239,-0.0021,0.0310
5,Perceptron,0.8163,0.9525,0.3590,0.8000,0.9377,0.4081,0.0163,0.0148,0.0491
6,Decision Tree,0.7636,0.8280,0.0254,0.7500,0.8134,0.0383,0.0136,0.0146,0.0129
7,Gradient Boosting,0.8400,0.9381,0.8859,0.8269,0.9482,1.5472,0.0131,-0.0101,0.6613
8,AdaBoost,0.8367,0.9487,1.9634,0.8283,0.9492,2.4756,0.0084,-0.0005,0.5122
9,Bagging,0.8155,0.9401,3.0513,0.8119,0.9287,4.5664,0.0036,0.0114,1.5151



✓ All done. Everything saved under: c:\Users\ssath\OneDrive\Documents\PCOS detection using ML\pcos-prediction-ml-clean\outputs\learning_curves
